# SBI 811 — Lecture 7

# Reading, Writing and Processing FASTA Files

## Computer Programming in Bioinformatics

Programming becomes especially useful when information is stored in files. A program can read hundreds or thousands of records, check their contents, calculate summaries and write new results without opening every file manually.

This lecture begins with ordinary text files and then applies the same ideas to FASTA files. FASTA is a widely used plain-text format for storing biological sequences.

The examples use only Python features already introduced in SBI 811:

- strings;
- lists;
- loops;
- `if`, `elif` and `else` statements;
- functions;
- dictionaries; and
- simple debugging and testing.

No pandas knowledge is required.

# Part 1 — Information in memory and information in files

When Python runs this instruction:

```python
sequence = "ATGCGT"
```

the string is stored in the computer's working memory. The variable normally disappears when the Python session ends.

A **file** stores information on a storage device. The information can remain available after Python stops. A later program can open the same file and read it again.

This distinction is important:

| Location | Example | Lifetime |
|---|---|---|
| Python memory | `sequence = "ATGCGT"` | Usually until the session ends |
| File storage | `sequences.fasta` | Remains until changed or deleted |

Reading a file transfers information from storage into Python. Writing a file transfers information from Python to storage.

# Part 2 — Understanding file paths

A **file path** describes the location of a file.

An absolute Linux path starts at the root directory `/`:

```text
/home/student/SBI811/lecture07_sequences.fasta
```

A relative path starts from the directory in which the notebook is running:

```text
lecture07_sequences.fasta
```

The notebook uses a relative path because the FASTA file should be saved in the same directory as the notebook.

Python can report its current working directory using the `Path` class from the `pathlib` module.

In [ ]:
from pathlib import Path

current_directory = Path.cwd()
print("Current directory:", current_directory)

### Explanation of the code

```python
from pathlib import Path
```

- `pathlib` is a Python module for working with paths.
- `Path` is one class contained in that module.
- `from ... import ...` makes the selected class available under its own name.

```python
current_directory = Path.cwd()
```

- `Path.cwd()` asks for the **current working directory**.
- The returned path is stored in `current_directory`.

```python
print("Current directory:", current_directory)
```

`print()` displays a label followed by the path.

## Check whether the supplied FASTA file is visible

Before opening a file, it is useful to check that Python can see it.

In [ ]:
fasta_path = Path("lecture07_sequences.fasta")

print("File exists:", fasta_path.exists())
print("Full path:", fasta_path.resolve())

`Path("lecture07_sequences.fasta")` creates a path object. It does not create or open the file.

`.exists()` returns a Boolean value:

- `True` means something exists at the path;
- `False` means Python cannot find it from the current location.

`.resolve()` displays an absolute form of the path. If `.exists()` returns `False`, check that the notebook and FASTA file are in the same directory and that the capitalization is identical.

Linux filenames are case-sensitive. These can be different files:

```text
sequences.fasta
Sequences.fasta
SEQUENCES.FASTA
```

# Part 3 — Opening and closing a text file

The built-in `open()` function connects Python to a file.

```python
file_object = open("lecture07_sequences.fasta", "r")
```

The first argument is the path. The second argument is the **mode**.

Common modes are:

| Mode | Meaning | Important behaviour |
|---|---|---|
| `"r"` | read | File must already exist |
| `"w"` | write | Creates a file or erases existing contents |
| `"a"` | append | Adds new text at the end |

After using a directly opened file, it should be closed:

```python
file_object.close()
```

Forgetting to close a file can delay writing, consume resources or leave the file in an uncertain state. Python therefore provides the `with` statement.

## The recommended form: `with open(...)`

In [ ]:
with open("lecture07_sequences.fasta", "r") as fasta_file:
    file_contents = fasta_file.read()

print(file_contents)

Read the first line in pieces:

```python
with open("lecture07_sequences.fasta", "r") as fasta_file:
```

- `open(...)` opens the named file in read mode.
- `as fasta_file` gives the opened file object a variable name.
- The colon `:` begins the indented `with` block.
- `fasta_file` represents the open connection, not the filename text.

Inside the block:

```python
file_contents = fasta_file.read()
```

`.read()` reads all remaining characters and returns one string. That string is stored in `file_contents`.

When the indented block finishes, Python closes the file automatically—even if an error occurs inside the block. The variable `file_contents` remains available because it contains the text already read into memory.

The name `fasta_file` may still exist after the block, but its file connection is closed and should no longer be read from.

# Part 4 — Three common reading methods

Python file objects provide several reading methods.

## 4.1 `.read()` — one string containing the remaining file

```python
with open("file.txt", "r") as file:
    text = file.read()
```

This is simple for a small file. A very large file may not fit comfortably in memory.

## 4.2 `.readline()` — one line

Each call reads the next line.

In [ ]:
with open("lecture07_sequences.fasta", "r") as fasta_file:
    first_line = fasta_file.readline()
    second_line = fasta_file.readline()
    third_line = fasta_file.readline()

print("First:", repr(first_line))
print("Second:", repr(second_line))
print("Third:", repr(third_line))

`repr()` displays a representation of a string that exposes special characters. The output may contain `\n` at the end of each line.

`\n` is the newline character. It records where one text line ends and the next begins. It is one character even though its representation uses a backslash and the letter `n`.

The file position advances after every call to `.readline()`. The first call returns line 1, the second returns line 2 and so on. At the end of the file, `.readline()` returns an empty string `""`.

## 4.3 `.readlines()` — a list of lines

In [ ]:
with open("lecture07_sequences.fasta", "r") as fasta_file:
    lines = fasta_file.readlines()

print("Type:", type(lines))
print("Number of lines:", len(lines))
print("First list item:", repr(lines[0]))

`.readlines()` returns a list. Each list item is one line, usually including its newline character.

This allows indexing and slicing:

```python
lines[0]      # first line
lines[-1]     # final line
lines[:3]     # first three lines
```

Like `.read()`, `.readlines()` loads the remaining file contents into memory. For large files, looping directly over the file is more memory-efficient.

# Part 5 — Reading a file one line at a time

An open text file is iterable. A `for` loop can request one line at a time.

In [ ]:
with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        print(repr(line))

The loop performs these steps repeatedly:

1. obtain the next line from `fasta_file`;
2. store that line temporarily in `line`;
3. execute the indented body;
4. stop automatically when no lines remain.

Only one line needs to be held for the loop operation at a time. This pattern is preferred for large sequence files.

The name `line` is not special. A descriptive alternative could be `fasta_line`. The chosen variable name should help the reader understand what one loop item represents.

# Part 6 — Removing newline characters

The newline is part of the string returned from a text file. It is often removed before analysing a line.

## `.strip()`

`.strip()` removes whitespace from both ends of a string. Whitespace includes spaces, tabs and newline characters.

## `.rstrip()`

`.rstrip()` removes whitespace only from the right-hand end.

## `.lstrip()`

`.lstrip()` removes whitespace only from the left-hand end.

In [ ]:
example_line = "  ATGCGT  \n"

print("Original:", repr(example_line))
print("strip():", repr(example_line.strip()))
print("rstrip():", repr(example_line.rstrip()))
print("lstrip():", repr(example_line.lstrip()))

For clean FASTA sequence lines, `.strip()` is commonly acceptable because surrounding whitespace is not part of the biological sequence.

For some other text formats, leading spaces may carry meaning. In that case, `.rstrip("\n")` can remove only the newline:

```python
line_without_newline = line.rstrip("\n")
```

This distinction matters:

- `.rstrip()` with no argument removes all types of trailing whitespace;
- `.rstrip("\n")` removes newline characters from the right;
- `.strip()` removes whitespace from both ends.

A method returns a new string. Strings are immutable, so the original string is not modified unless the returned value is assigned.

In [ ]:
raw_line = "ATGCGT\n"
raw_line.strip()

print("Without assignment:", repr(raw_line))

clean_line = raw_line.strip()
print("Stored result:", repr(clean_line))

Calling `raw_line.strip()` calculates a cleaned string, but the result is lost if it is not stored or used. The assignment to `clean_line` preserves the returned string.

# Part 7 — The FASTA format

A FASTA file stores one or more sequence records as plain text.

A record contains:

1. a header line beginning with `>`;
2. one or more sequence lines.

Example:

```text
>seq01 example sequence
ATGCGTACGTTAGC
```

The greater-than symbol `>` identifies a header. The text immediately following it often contains a sequence identifier. Additional description may follow.

A multi-record FASTA file can look like:

```text
>seq01 first sequence
ATGCGT
AACCGG
>seq02 second sequence
TTGCAA
```

Important observations:

- a sequence may occupy one line or several lines;
- a new record begins when another header is encountered;
- blank lines are best ignored or treated according to a defined rule;
- nucleotide sequences may use uppercase or lowercase letters;
- ambiguous symbols such as `N` may occur;
- the FASTA format itself does not guarantee that sequence characters are valid.

A parser must not assume that one sequence always occupies exactly one line.

# Part 8 — Identifying headers and sequence lines

The string method `.startswith()` checks the beginning of a string.

In [ ]:
with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()

        if clean_line.startswith(">"):
            print("HEADER:", clean_line)
        else:
            print("SEQUENCE:", clean_line)

The condition:

```python
clean_line.startswith(">")
```

returns `True` when the first character is `>` and `False` otherwise.

The `if` block runs for headers. The `else` block runs for every line that is not a header. This first version would label a blank line as a sequence line. A more careful version tests for blank lines explicitly.

In [ ]:
with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()

        if clean_line == "":
            continue
        elif clean_line.startswith(">"):
            print("HEADER:", clean_line)
        else:
            print("SEQUENCE:", clean_line)

`clean_line == ""` asks whether the cleaned line is an empty string.

`continue` immediately moves to the next loop iteration. No later code in the current iteration is executed.

The order of the conditions is deliberate:

1. ignore blank lines;
2. identify headers;
3. treat remaining non-empty lines as sequence lines.

This still does not prove that a remaining line contains a valid sequence. Validation is added later.

# Part 9 — Reading a single FASTA record

Begin with the simpler case of one header and one or more sequence lines.

The goal is to produce:

```python
identifier = "seq01"
sequence = "ATGCGTAACCGG"
```

Sequence fragments are collected in a list and joined after reading.

In [ ]:
identifier = ""
sequence_parts = []

with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()

        if clean_line == "":
            continue

        if clean_line.startswith(">"):
            header_without_symbol = clean_line[1:]
            identifier = header_without_symbol.split()[0]
        else:
            sequence_parts.append(clean_line)

sequence = "".join(sequence_parts)

print("Last identifier read:", identifier)
print("All sequence text joined:", sequence)

This code demonstrates the mechanics, but it is **not a correct multi-record parser**. Because the supplied file has several records, all sequence lines are joined together and `identifier` is repeatedly replaced. The final identifier is associated incorrectly with the combined sequence.

That incorrect result teaches an important lesson: code can run without an error and still produce scientifically incorrect output.

### Syntax used

```python
clean_line[1:]
```

This slice starts at index 1 and continues to the end, thereby removing the first `>` character.

```python
header_without_symbol.split()[0]
```

- `.split()` divides the header at whitespace and returns a list;
- `[0]` selects the first item as the identifier.

```python
sequence_parts.append(clean_line)
```

adds one sequence fragment to the list.

```python
"".join(sequence_parts)
```

joins all list items with an empty separator. Joining once is efficient and clearly expresses the intention to combine sequence lines.

# Part 10 — Designing a multi-record FASTA parser

A multi-record parser must know when one record finishes and the next begins.

Useful state variables are:

- `current_id` — identifier of the record currently being collected;
- `current_parts` — list of sequence lines for that record;
- `sequences` — dictionary containing completed records.

When a new header appears:

1. save the previous record if one exists;
2. extract the new identifier;
3. reset the sequence-parts list.

After the loop, save the final record because no later header appears to trigger its storage.

## Pseudocode

```text
create an empty dictionary
set current identifier to nothing
create an empty list of sequence parts

open the FASTA file
for each line:
    remove surrounding whitespace
    if the line is blank:
        skip it
    if the line is a header:
        if a previous identifier exists:
            join and store its sequence parts
        extract the new identifier
        reset the sequence-parts list
    otherwise:
        add the line to the current sequence parts

after the loop:
    store the final record
```

In [ ]:
sequences = {}
current_id = None
current_parts = []

with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()

        if clean_line == "":
            continue

        if clean_line.startswith(">"):
            if current_id is not None:
                complete_sequence = "".join(current_parts)
                sequences[current_id] = complete_sequence

            header_text = clean_line[1:]
            current_id = header_text.split()[0]
            current_parts = []
        else:
            current_parts.append(clean_line)

if current_id is not None:
    complete_sequence = "".join(current_parts)
    sequences[current_id] = complete_sequence

print(sequences)

### Understanding the state variables

```python
sequences = {}
```

creates an empty dictionary. Identifiers will become keys and sequence strings will become values.

```python
current_id = None
```

`None` represents the absence of a current identifier before the first header is read. It is different from the text string `"None"`.

```python
current_parts = []
```

creates an empty list for the current record's sequence lines.

### Why test `current_id is not None`?

When the first header appears, there is no earlier record to save. The test prevents the parser from creating an invalid dictionary entry before a genuine identifier exists.

### Why reset `current_parts`?

After saving the previous sequence, a new empty list must be created. Otherwise, fragments from the previous record would remain and become part of the next sequence.

### Why save once after the loop?

Records are saved when the next header arrives. The final record has no next header, so it must be stored after the loop ends. Forgetting this step is a common parsing error that silently loses the final record.

## Inspect parsed records clearly

In [ ]:
print("Number of records:", len(sequences))

for sequence_id, sequence in sequences.items():
    print(sequence_id, "length =", len(sequence))

`.items()` produces dictionary key–value pairs. During each iteration:

- `sequence_id` receives one key;
- `sequence` receives the associated value.

`len(sequences)` counts dictionary entries. `len(sequence)` counts characters in the sequence string.

The dictionary allows direct lookup:

```python
sequences["seq03"]
```

If the key does not exist, square-bracket lookup raises `KeyError`. `.get()` can return `None` or a chosen default instead:

```python
sequences.get("seq99")
sequences.get("seq99", "not found")
```

In [ ]:
print("seq03:", sequences.get("seq03"))
print("seq99:", sequences.get("seq99", "not found"))

# Part 11 — Normalizing sequence case

Uppercase and lowercase nucleotide letters normally represent the same nucleotide. Standardizing case makes later comparisons easier.

In [ ]:
for sequence_id in sequences:
    sequences[sequence_id] = sequences[sequence_id].upper()

print(sequences["seq03"])

The loop iterates over dictionary keys. For each key:

1. retrieve the sequence value;
2. call `.upper()` to return an uppercase string;
3. assign that string back to the same key.

Because strings are immutable, `.upper()` does not alter the old string. The assignment stores the returned version.

Case normalization is different from validation. Converting `x` to `X` does not make it a valid nucleotide symbol.

# Part 12 — Validating nucleotide sequences

A simple DNA alphabet may allow:

```text
A C G T
```

Many real FASTA files also use `N` for an unknown or ambiguous nucleotide. The correct allowed alphabet depends on the data and scientific purpose.

For this lecture, the allowed set is:

```python
allowed_bases = {"A", "C", "G", "T", "N"}
```

A set is useful for membership testing.

In [ ]:
allowed_bases = {"A", "C", "G", "T", "N"}

for sequence_id, sequence in sequences.items():
    invalid_bases = set(sequence) - allowed_bases

    if len(invalid_bases) == 0:
        print(sequence_id, "is valid")
    else:
        print(sequence_id, "contains invalid symbols:", invalid_bases)

`set(sequence)` creates a set of the distinct characters found in the sequence.

The subtraction:

```python
set(sequence) - allowed_bases
```

keeps characters that occur in the sequence but not in the allowed set.

If the resulting set has length zero, no invalid symbols were found.

This validation checks characters only. It does not prove that:

- the sequence belongs to the claimed organism;
- the identifier is correct;
- the molecule is complete;
- the orientation is correct; or
- the sequence is free from experimental errors.

## Package validation inside a function

In [ ]:
def find_invalid_bases(sequence, allowed_bases={"A", "C", "G", "T", "N"}):
    """Return the set of sequence symbols that are not allowed."""
    sequence_upper = sequence.upper()
    observed_bases = set(sequence_upper)
    invalid_bases = observed_bases - allowed_bases
    return invalid_bases


print(find_invalid_bases("ATGCNN"))
print(find_invalid_bases("ATGCX"))

The function has two parameters:

- `sequence` receives the sequence to inspect;
- `allowed_bases` has a default set.

The docstring explains what the function returns. Returning the invalid set provides more information than returning only `True` or `False`.

The default set is not modified by the function. Mutable default arguments can cause problems when a function changes them; this function only reads the set.

# Part 13 — Calculating GC percentage

GC percentage describes the proportion of `G` and `C` characters.

For a sequence of length (L):

$$
GC\% = \frac{G + C}{L} \times 100
$$

A zero-length sequence requires special handling because division by zero is not allowed.

In [ ]:
def gc_percentage(sequence):
    """Return the GC percentage of a nucleotide sequence."""
    sequence_upper = sequence.upper()

    if len(sequence_upper) == 0:
        return 0.0

    g_count = sequence_upper.count("G")
    c_count = sequence_upper.count("C")
    gc_count = g_count + c_count

    percentage = (gc_count / len(sequence_upper)) * 100
    return percentage

The function:

1. normalizes case;
2. checks for an empty string;
3. counts `G` and `C` separately;
4. adds the counts;
5. divides by sequence length;
6. multiplies by 100;
7. returns the result.

The early `return 0.0` ends the function for an empty sequence and prevents later division by zero.

In [ ]:
for sequence_id, sequence in sequences.items():
    gc = gc_percentage(sequence)
    print(sequence_id, "GC% =", round(gc, 2))

`round(gc, 2)` returns the value rounded to two decimal places for display. It does not change how the original sequence was counted.

How ambiguous bases should affect the denominator is a scientific decision. The function above includes `N` in total sequence length. Another clearly documented definition might calculate GC percentage only among `A`, `C`, `G` and `T` bases. The definition must be stated rather than assumed.

# Part 14 — Writing text files

Write mode `"w"` creates a file or replaces its existing contents.

In [ ]:
with open("lecture07_summary.txt", "w") as output_file:
    output_file.write("Sequence summary\n")
    output_file.write("Number of records: " + str(len(sequences)) + "\n")

print("Summary file written.")

`.write()` expects a string. `len(sequences)` produces an integer, so `str(...)` converts it before string concatenation.

Unlike `print()`, `.write()` does not automatically add a newline. The `\n` characters are included explicitly.

> **Important:** Opening an existing file with `"w"` immediately truncates it to zero length. Confirm the output path before running the cell.

Read the saved file back rather than assuming that writing succeeded.

In [ ]:
with open("lecture07_summary.txt", "r") as output_file:
    saved_summary = output_file.read()

print(saved_summary)

## Append mode

Append mode `"a"` adds text at the end of a file.

In [ ]:
with open("lecture07_summary.txt", "a") as output_file:
    output_file.write("Processing completed.\n")

Running an append cell repeatedly adds the same line repeatedly. Append mode is useful for logs, but repeated notebook execution must be considered.

For reproducible analysis outputs, write mode is often preferable because each run creates a fresh, complete result rather than accumulating old and new lines.

# Part 15 — Writing FASTA records

A FASTA writer must produce a header beginning with `>` followed by sequence text.

This example writes one sequence per line.

In [ ]:
with open("lecture07_uppercase_sequences.fasta", "w") as output_fasta:
    for sequence_id, sequence in sequences.items():
        output_fasta.write(">" + sequence_id + "\n")
        output_fasta.write(sequence + "\n")

print("FASTA file written.")

For each dictionary entry:

1. `">" + sequence_id + "\n"` constructs the header line;
2. `sequence + "\n"` constructs the sequence line;
3. each string is written to the open output file.

The dictionary stores identifiers but not the original descriptive header text. Therefore, this output contains only identifiers. Preserving descriptions would require storing them separately or storing richer records.

In [ ]:
with open("lecture07_uppercase_sequences.fasta", "r") as output_fasta:
    print(output_fasta.read())

# Part 16 — Wrapping long FASTA sequences

Long sequences are often written across fixed-width lines, commonly 60 or 80 characters. String slicing can create these lines.

In [ ]:
example_sequence = "ATGCGT" * 15
line_width = 20

for start in range(0, len(example_sequence), line_width):
    sequence_line = example_sequence[start:start + line_width]
    print(sequence_line)

`range(0, len(example_sequence), line_width)` produces starting positions:

```text
0, 20, 40, 60, ...
```

The three `range()` arguments are:

1. start at 0;
2. stop before sequence length;
3. increase by `line_width`.

The slice:

```python
example_sequence[start:start + line_width]
```

selects a segment beginning at `start`. The last segment may be shorter than the chosen width.

In [ ]:
def write_fasta(sequences, output_path, line_width=60):
    """Write an ID-to-sequence dictionary in FASTA format."""
    with open(output_path, "w") as output_fasta:
        for sequence_id, sequence in sequences.items():
            output_fasta.write(">" + sequence_id + "\n")

            for start in range(0, len(sequence), line_width):
                sequence_line = sequence[start:start + line_width]
                output_fasta.write(sequence_line + "\n")

The function receives:

- `sequences` — a dictionary of identifiers and sequence strings;
- `output_path` — destination filename or path;
- `line_width=60` — an optional line width with a default value.

The outer loop visits records. The inner loop visits chunks within one sequence.

In [ ]:
write_fasta(
    sequences,
    "lecture07_wrapped_sequences.fasta",
    line_width=15
)

with open("lecture07_wrapped_sequences.fasta", "r") as check_file:
    print(check_file.read())

# Part 17 — Packaging the FASTA reader as a function

The earlier parser can become a reusable function.

In [ ]:
def read_fasta(fasta_path):
    """Read a FASTA file and return an ID-to-sequence dictionary."""
    sequences = {}
    current_id = None
    current_parts = []

    with open(fasta_path, "r") as fasta_file:
        for line in fasta_file:
            clean_line = line.strip()

            if clean_line == "":
                continue

            if clean_line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_parts).upper()

                header_text = clean_line[1:]
                current_id = header_text.split()[0]
                current_parts = []
            else:
                current_parts.append(clean_line)

    if current_id is not None:
        sequences[current_id] = "".join(current_parts).upper()

    return sequences

The function contains the same algorithm already examined. Packaging it provides several benefits:

- it can be called for different file paths;
- the parsing details do not need to be copied repeatedly;
- it can be tested independently;
- its returned dictionary can be passed to other functions.

This function is intentionally small and educational. Mature libraries such as Biopython provide more comprehensive FASTA parsing, but understanding the basic algorithm helps explain what a parser must do.

In [ ]:
loaded_sequences = read_fasta("lecture07_sequences.fasta")

print("Records loaded:", len(loaded_sequences))
print("Identifiers:", list(loaded_sequences.keys()))

`loaded_sequences.keys()` returns a dictionary view of keys. `list(...)` converts that view to a list for display.

# Part 18 — Detecting structural FASTA problems

The simple parser accepts several problematic situations unless checks are added:

- sequence text before the first header;
- an empty header;
- duplicate identifiers;
- a header without sequence data;
- invalid sequence characters.

Silent acceptance can cause records to be overwritten or misassigned.

## Duplicate identifiers

A dictionary key must be unique. If the same identifier appears twice, later assignment replaces the earlier value. A parser should detect the duplication before assignment.

```python
if new_id in sequences:
    raise ValueError("Duplicate FASTA identifier: " + new_id)
```

`raise` stops normal execution by creating an exception. `ValueError` indicates that a value does not satisfy an expected rule.

## Sequence before a header

```python
if current_id is None:
    raise ValueError("Sequence found before the first FASTA header")
```

Clear failures are safer than silently producing incorrect data.

# Part 19 — Building a summary table without pandas

The output will be a tab-separated text file with four columns:

```text
sequence_id    length    gc_percent    invalid_symbols
```

In [ ]:
with open("lecture07_sequence_report.tsv", "w") as report_file:
    report_file.write("sequence_id\tlength\tgc_percent\tinvalid_symbols\n")

    for sequence_id, sequence in loaded_sequences.items():
        length = len(sequence)
        gc = round(gc_percentage(sequence), 2)
        invalid = find_invalid_bases(sequence)

        if len(invalid) == 0:
            invalid_text = "none"
        else:
            invalid_text = ",".join(sorted(invalid))

        row = (
            sequence_id + "\t" +
            str(length) + "\t" +
            str(gc) + "\t" +
            invalid_text + "\n"
        )

        report_file.write(row)

### Understanding the report code

The first `.write()` call creates a header row. `\t` represents a tab character.

For every FASTA record, the program calculates summary values. All values must become strings before they are joined with `+`.

If invalid symbols exist:

```python
",".join(sorted(invalid))
```

sorts the set and joins the symbols using commas. Sorting makes the output order stable.

Parentheses allow the long `row` expression to span several physical lines without changing the resulting string.

In [ ]:
with open("lecture07_sequence_report.tsv", "r") as report_file:
    print(report_file.read())

# Part 20 — Handling file errors

Opening a missing file raises `FileNotFoundError`.

```python
try:
    with open("missing.fasta", "r") as fasta_file:
        text = fasta_file.read()
except FileNotFoundError:
    print("The FASTA file was not found.")
```

`try` identifies code that may raise an exception. `except FileNotFoundError` handles that specific error.

Do not use a bare `except:` around large sections of code. It can hide programming mistakes and make debugging difficult.

A function can also check a path before opening:

```python
path = Path("missing.fasta")

if not path.exists():
    print("Missing file:", path)
```

Checking first can produce a friendly message, but a file can still become unavailable between checking and opening. Exception handling remains important in robust software.

# Part 21 — Testing the functions

Small known examples make mistakes easier to detect.

In [ ]:
assert gc_percentage("GGCC") == 100.0
assert gc_percentage("AATT") == 0.0
assert gc_percentage("") == 0.0
assert find_invalid_bases("ATGCN") == set()
assert find_invalid_bases("ATGX") == {"X"}

print("Function tests passed.")

`assert condition` raises `AssertionError` when the condition is false. No error means the tested examples behaved as expected.

These tests do not prove that the functions are correct for every possible input. They provide evidence for selected cases and can detect regressions after code changes.

Useful additional cases include:

- lowercase sequences;
- a sequence containing only `N`;
- a very long sequence;
- a one-character sequence;
- unexpected whitespace;
- duplicate FASTA identifiers;
- an empty FASTA file.

# Part 22 — Integrated FASTA-processing workflow

The complete workflow is:

```text
locate input file
        ↓
read FASTA records
        ↓
normalize sequence case
        ↓
validate characters
        ↓
calculate summaries
        ↓
write report and FASTA output
        ↓
reopen outputs and verify them
```

Each stage has a distinct responsibility. Keeping stages separate makes it easier to locate errors.

In [ ]:
input_path = "lecture07_sequences.fasta"
output_path = "lecture07_valid_sequences.fasta"

all_sequences = read_fasta(input_path)
valid_sequences = {}

for sequence_id, sequence in all_sequences.items():
    invalid_bases = find_invalid_bases(sequence)

    if len(invalid_bases) == 0:
        valid_sequences[sequence_id] = sequence
    else:
        print(sequence_id, "was excluded because of:", invalid_bases)

write_fasta(valid_sequences, output_path, line_width=60)

print("Input records:", len(all_sequences))
print("Valid records written:", len(valid_sequences))

In this lecture's allowed alphabet, `N` is accepted. Therefore, `seq05` remains valid. If a strict `A`, `C`, `G`, `T` alphabet were required, the allowed set passed to the validation function would need to change explicitly.

Filtering is never scientifically neutral. Before excluding a record, document:

- the validation rule;
- why the rule is appropriate;
- which records were excluded;
- whether the original data remain preserved.

# Part 23 — Guided exercises

## Exercise 1: inspect the FASTA file

Write code that:

1. opens `lecture07_sequences.fasta` safely;
2. counts all physical lines;
3. counts header lines;
4. counts non-empty sequence lines;
5. prints the three counts.

### Pseudocode

```text
set three counters to zero
open the FASTA file
for every line:
    increase the physical-line counter
    remove surrounding whitespace
    if the line begins with >:
        increase the header counter
    else if the line is not empty:
        increase the sequence-line counter
print all counters
```

In [ ]:
# Write Exercise 1 here.

### Exercise 1 — one possible solution

In [ ]:
physical_lines = 0
header_lines = 0
sequence_lines = 0

with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        physical_lines += 1
        clean_line = line.strip()

        if clean_line.startswith(">"):
            header_lines += 1
        elif clean_line != "":
            sequence_lines += 1

print("Physical lines:", physical_lines)
print("Header lines:", header_lines)
print("Sequence lines:", sequence_lines)

`+= 1` is shorthand for adding one to the existing value. The `elif` prevents header lines from also being counted as sequence lines.

## Exercise 2: longest sequence

Use the parsed dictionary to identify the longest sequence. Print its identifier and length.

### Pseudocode

```text
set longest identifier to nothing
set longest length to zero
for each identifier and sequence:
    calculate its length
    if the length is greater than the longest length:
        save this identifier
        save this length
print the saved identifier and length
```

In [ ]:
# Write Exercise 2 here.

### Exercise 2 — one possible solution

In [ ]:
longest_id = None
longest_length = 0

for sequence_id, sequence in loaded_sequences.items():
    sequence_length = len(sequence)

    if sequence_length > longest_length:
        longest_id = sequence_id
        longest_length = sequence_length

print("Longest identifier:", longest_id)
print("Longest length:", longest_length)

The stored values change only when a strictly longer sequence is found. If two sequences tie, this version retains the first one encountered. Tie handling should be changed if all longest records are required.

## Exercise 3: select sequences by length

Create a new dictionary containing sequences whose length is at least 35 nucleotides. Write the selected sequences to `lecture07_long_sequences.fasta`.

### Pseudocode

```text
create an empty dictionary
for each parsed sequence:
    if its length is at least 35:
        add it to the new dictionary
write the new dictionary as FASTA
reopen the output and inspect it
```

In [ ]:
# Write Exercise 3 here.

### Exercise 3 — one possible solution

In [ ]:
long_sequences = {}

for sequence_id, sequence in loaded_sequences.items():
    if len(sequence) >= 35:
        long_sequences[sequence_id] = sequence

write_fasta(
    long_sequences,
    "lecture07_long_sequences.fasta",
    line_width=60
)

with open("lecture07_long_sequences.fasta", "r") as check_file:
    print(check_file.read())

# Part 24 — Independent practice

Create a Python program that processes a supplied FASTA file and produces two outputs.

## Output 1: sequence report

The TSV report should contain:

- identifier;
- length;
- count of `A`;
- count of `C`;
- count of `G`;
- count of `T`;
- count of `N`;
- GC percentage;
- validation status.

## Output 2: filtered FASTA

Write only records that:

- contain no invalid symbols under the chosen alphabet;
- are at least 20 nucleotides long.

## Required safeguards

The program should:

1. preserve the input file;
2. use `with open(...)`;
3. give clear output filenames;
4. check that the input exists;
5. document whether `N` is allowed;
6. test at least two helper functions with `assert`;
7. reopen the outputs and verify their record or line counts.

Begin with pseudocode before writing Python.

In [ ]:
# Develop the independent practice program here.

# Part 25 — Common mistakes

## Mistake 1: opening the input with `"w"`

```python
open("important_input.fasta", "w")
```

This can erase the file immediately. Input files should normally be opened with `"r"`.

## Mistake 2: assuming one sequence line per record

A valid FASTA sequence can span several lines. Join all sequence lines until the next header.

## Mistake 3: forgetting the final record

If records are saved only when a new header appears, the last record must be saved after the loop.

## Mistake 4: not resetting sequence parts

Failing to reset the list causes one record's sequence to be included in the next.

## Mistake 5: losing the result of `.strip()`

```python
line.strip()
```

does not modify `line`. Store the result:

```python
clean_line = line.strip()
```

## Mistake 6: accepting duplicate identifiers silently

A dictionary assignment can overwrite the earlier sequence. Check duplicates before storing.

## Mistake 7: treating successful execution as proof of correctness

A program may run without exceptions and still combine the wrong records. Test with a small file whose correct result is known.

# Part 26 — Key terms

| Term | Meaning |
|---|---|
| File path | Text or path object describing a file location |
| Relative path | Location interpreted from the current working directory |
| Absolute path | Complete location beginning at the filesystem root |
| File object | Python object representing an open connection to a file |
| Read mode | Mode `"r"`; obtains existing file contents |
| Write mode | Mode `"w"`; creates or replaces file contents |
| Append mode | Mode `"a"`; adds text to the end |
| Newline | Character `\n` marking the end of a text line |
| FASTA header | Line beginning with `>` |
| FASTA record | Header plus its sequence lines |
| Parser | Code that reads a format and converts it into program data |
| Validation | Checking data against explicit rules |
| Normalization | Converting equivalent representations to one standard form |
| Exception | Object representing an error or exceptional event |
| TSV | Tab-separated values text file |

# Lecture summary

This lecture developed a complete path from ordinary text files to reusable FASTA processing.

The main ideas are:

- a file stores information beyond the current Python session;
- paths tell Python where files are located;
- `with open(...)` manages file closing safely;
- `.read()`, `.readline()`, `.readlines()` and direct iteration serve different purposes;
- newline characters are part of text read from files;
- `.strip()` returns a new string;
- FASTA records begin with `>` headers and may contain several sequence lines;
- a multi-record parser must save the previous record and the final record;
- dictionaries connect sequence identifiers with sequences;
- normalization and validation are different operations;
- write mode can erase existing contents;
- outputs should be reopened and checked;
- small functions and tests make a workflow easier to understand and maintain.

Lecture 8 can build on this foundation by performing a larger sequence-analysis workflow or by introducing additional command-line processing, depending on the course plan.